In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

# === CONFIG ===
CSV_PATH = "/content/drive/Shareddrives/Lopez_Morrison_Summer25/code_counts/code_counts.csv"   # change if needed
METRIC   = "num_snippets"    # <- set any numeric metric column here
GROUP_BYS = ["method", "model", "subject"]  # attributes to split by
MIN_GROUP_N = 3                # minimum rows required to compute correlations


def _coerce_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _corr_pair(x: pd.Series, y: pd.Series):
    """
    Returns (pearson_r, pearson_p, spearman_rho, spearman_p) or (np.nan, ... )
    Handles edge cases: fewer than 2 rows, zero variance, NaNs.
    """
    # drop NaNs aligned
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(df) < 2:
        return (np.nan, np.nan, np.nan, np.nan)

    # zero variance -> correlation undefined
    if df["x"].nunique() < 2 or df["y"].nunique() < 2:
        return (np.nan, np.nan, np.nan, np.nan)

    try:
        pr, pp = pearsonr(df["x"], df["y"])
    except Exception:
        pr, pp = (np.nan, np.nan)

    try:
        srho, sp = spearmanr(df["x"], df["y"])
    except Exception:
        srho, sp = (np.nan, np.nan)

    return (pr, pp, srho, sp)


def compute_overall(df: pd.DataFrame, metric: str):
    pr, pp, sr, sp = _corr_pair(df["question_number"], df[metric])
    return [{
        "scope": "overall",
        "group_by": "",
        "group_value": "",
        "n": int(df.dropna(subset=["question_number", metric]).shape[0]),
        "pearson_r": pr, "pearson_p": pp,
        "spearman_rho": sr, "spearman_p": sp
    }]


def compute_by(df: pd.DataFrame, metric: str, group_col: str, min_n: int):
    rows = []
    for group_val, g in df.groupby(group_col, dropna=False):
        g_clean = g.dropna(subset=["question_number", metric])
        if len(g_clean) < min_n:
            rows.append({
                "scope": "by_"+group_col,
                "group_by": group_col,
                "group_value": "" if pd.isna(group_val) else str(group_val),
                "n": int(len(g_clean)),
                "pearson_r": np.nan, "pearson_p": np.nan,
                "spearman_rho": np.nan, "spearman_p": np.nan,
                "note": f"insufficient_n(<{min_n})"
            })
            continue

        pr, pp, sr, sp = _corr_pair(g_clean["question_number"], g_clean[metric])
        note = ""
        if np.isnan(pr) and np.isnan(sr):
            # likely zero variance or ties only
            note = "undefined_corr(variance/ties)"

        rows.append({
            "scope": "by_"+group_col,
            "group_by": group_col,
            "group_value": "" if pd.isna(group_val) else str(group_val),
            "n": int(len(g_clean)),
            "pearson_r": pr, "pearson_p": pp,
            "spearman_rho": sr, "spearman_p": sp,
            "note": note
        })
    return rows


def main(csv_path: str, metric: str, group_bys, min_group_n: int):
    df = pd.read_csv(csv_path)

    # Coerce needed columns
    needed_cols = ["question_number", metric]
    for col in needed_cols:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in {csv_path}")

    df["question_number"] = _coerce_numeric(df["question_number"])
    df[metric] = _coerce_numeric(df[metric])

    # Compute correlations
    results = []
    results += compute_overall(df, metric)

    for gb in group_bys:
        if gb in df.columns:
            results += compute_by(df, metric, gb, min_group_n)
        else:
            results.append({
                "scope": "by_"+gb,
                "group_by": gb,
                "group_value": "",
                "n": 0,
                "pearson_r": np.nan, "pearson_p": np.nan,
                "spearman_rho": np.nan, "spearman_p": np.nan,
                "note": "group_by_column_missing"
            })

    out = pd.DataFrame(results, columns=[
        "scope","group_by","group_value","n",
        "pearson_r","pearson_p","spearman_rho","spearman_p","note"
    ])

    # Nice sorting: overall first, then by group, descending n
    out["__sort_scope"] = out["scope"].apply(lambda s: 0 if s=="overall" else 1)
    out = out.sort_values(["__sort_scope","group_by","n"], ascending=[True, True, False]).drop(columns="__sort_scope")

    # Save
    out_path = f"metric_correlations_{metric}.csv"
    out.to_csv(out_path, index=False)

    # Print a concise view
    print(f"\n=== Correlations vs. question_number for metric: {metric} ===")
    print(out.fillna("").to_string(index=False))
    print(f"\nSaved: {out_path}")


if __name__ == "__main__":
    # Run with defaults above; optionally replace METRIC at the top.
    main(CSV_PATH, METRIC, GROUP_BYS, MIN_GROUP_N)

In [ ]:
# correlation_scan.py
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

# =========================
# CONFIG
# =========================
CSV_PATH = "/content/drive/Shareddrives/Lopez_Morrison_Summer25/code_counts/code_counts.csv"

# Metrics to scan — edit this list as you wish.
# Any that are missing in the CSV will be skipped automatically.
METRICS = [
    "total_newlines",
    "cf_keyword_count",
    "max_nesting_depth",
    "token_count",
    "avg_tokens_per_code_line",
    "operators_count",
    "operators_per_token_ratio",
    "avg_line_length",
    "max_line_length",
    "total_non_ws_chars",
    "total_chars",
    "total_python_keywords",
    "num_snippets",
]

GROUP_BYS = ["method", "model", "subject"]  # attributes to split by
MIN_GROUP_N = 3                              # min rows to compute group-wise correlations

# What counts as “strong & significant” for highlighting
R_STRONG = 0.50
P_THRESH = 0.05


# =========================
# Helpers
# =========================
def _coerce_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _corr_pair(x: pd.Series, y: pd.Series):
    """
    Returns (pearson_r, pearson_p, spearman_rho, spearman_p) or (np.nan, ... )
    Handles: NaNs, N<2, zero-variance, exceptions.
    """
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(df) < 2 or df["x"].nunique() < 2 or df["y"].nunique() < 2:
        return (np.nan, np.nan, np.nan, np.nan)

    try:
        pr, pp = pearsonr(df["x"], df["y"])
    except Exception:
        pr, pp = (np.nan, np.nan)

    try:
        srho, sp = spearmanr(df["x"], df["y"])
    except Exception:
        srho, sp = (np.nan, np.nan)

    return (pr, pp, srho, sp)


def _compute_overall(df: pd.DataFrame, metric: str):
    pr, pp, sr, sp = _corr_pair(df["question_number"], df[metric])
    return [{
        "metric": metric,
        "scope": "overall",
        "group_by": "",
        "group_value": "",
        "n": int(df.dropna(subset=["question_number", metric]).shape[0]),
        "pearson_r": pr, "pearson_p": pp,
        "spearman_rho": sr, "spearman_p": sp,
        "strong_sig": _strong_sig_flag(pr, pp, sr, sp),
    }]


def _compute_by(df: pd.DataFrame, metric: str, group_col: str, min_n: int):
    rows = []
    for group_val, g in df.groupby(group_col, dropna=False):
        g_clean = g.dropna(subset=["question_number", metric])
        if len(g_clean) < min_n:
            rows.append({
                "metric": metric,
                "scope": f"by_{group_col}",
                "group_by": group_col,
                "group_value": "" if pd.isna(group_val) else str(group_val),
                "n": int(len(g_clean)),
                "pearson_r": np.nan, "pearson_p": np.nan,
                "spearman_rho": np.nan, "spearman_p": np.nan,
                "strong_sig": "",
                "note": f"insufficient_n(<{min_n})",
            })
            continue

        pr, pp, sr, sp = _corr_pair(g_clean["question_number"], g_clean[metric])
        note = ""
        if np.isnan(pr) and np.isnan(sr):
            note = "undefined_corr(variance/ties)"

        rows.append({
            "metric": metric,
            "scope": f"by_{group_col}",
            "group_by": group_col,
            "group_value": "" if pd.isna(group_val) else str(group_val),
            "n": int(len(g_clean)),
            "pearson_r": pr, "pearson_p": pp,
            "spearman_rho": sr, "spearman_p": sp,
            "strong_sig": _strong_sig_flag(pr, pp, sr, sp),
            "note": note,
        })
    return rows


def _strong_sig_flag(pr, pp, sr, sp):
    flags = []
    if not np.isnan(pr) and not np.isnan(pp) and (pr >= R_STRONG) and (pp < P_THRESH):
        flags.append("Pearson++")
    if not np.isnan(sr) and not np.isnan(sp) and (sr >= R_STRONG) and (sp < P_THRESH):
        flags.append("Spearman++")
    return "|".join(flags)


# =========================
# Main
# =========================
def main():
    df = pd.read_csv(CSV_PATH)

    # Ensure columns exist and are numeric
    if "question_number" not in df.columns:
        raise ValueError("Column 'question_number' not found in CSV.")
    df["question_number"] = _coerce_numeric(df["question_number"])

    # Keep only metrics that actually exist
    metrics_present = [m for m in METRICS if m in df.columns]
    if not metrics_present:
        raise ValueError("None of the specified METRICS were found in the CSV.")

    # Coerce metric columns to numeric
    for m in metrics_present:
        df[m] = _coerce_numeric(df[m])

    all_rows = []

    for metric in metrics_present:
        # overall
        all_rows += _compute_overall(df, metric)
        # by method/model/subject
        for gb in GROUP_BYS:
            if gb in df.columns:
                all_rows += _compute_by(df, metric, gb, MIN_GROUP_N)
            else:
                all_rows.append({
                    "metric": metric,
                    "scope": f"by_{gb}",
                    "group_by": gb,
                    "group_value": "",
                    "n": 0,
                    "pearson_r": np.nan, "pearson_p": np.nan,
                    "spearman_rho": np.nan, "spearman_p": np.nan,
                    "strong_sig": "",
                    "note": "group_by_column_missing",
                })

    out = pd.DataFrame(all_rows, columns=[
        "metric","scope","group_by","group_value","n",
        "pearson_r","pearson_p","spearman_rho","spearman_p",
        "strong_sig","note"
    ])

    # Sort: overall first, then by group, then metric, then n desc
    out["__scope_order"] = out["scope"].apply(lambda s: 0 if s == "overall" else 1)
    out = out.sort_values(["__scope_order", "metric", "group_by", "n"], ascending=[True, True, True, False]) \
             .drop(columns="__scope_order")

    summary_path = "metric_correlations_summary.csv"
    out.to_csv(summary_path, index=False)

    # =========================
    # Leaderboards (printed)
    # =========================
    def _top(df_in, col_r, label, k=10):
        dfv = df_in.dropna(subset=[col_r]).copy()
        # Positive direction only (we care about strongest positive correlations)
        dfv = dfv[dfv[col_r] > 0]
        return dfv.sort_values(col_r, ascending=False).head(k)

    overall = out[out["scope"] == "overall"]
    grouped = out[out["scope"] != "overall"]

    top_overall_pearson = _top(overall, "pearson_r", "overall pearson")
    top_overall_spearman = _top(overall, "spearman_rho", "overall spearman")
    top_grouped_pearson = _top(grouped, "pearson_r", "grouped pearson")
    top_grouped_spearman = _top(grouped, "spearman_rho", "grouped spearman")

    pd.options.display.float_format = '{:,.3f}'.format

    print(f"\nSaved: {summary_path}")
    print("\n=== TOP OVERALL (Pearson r) ===")
    if not top_overall_pearson.empty:
        print(top_overall_pearson[["metric","n","pearson_r","pearson_p","strong_sig"]].to_string(index=False))
    else:
        print("(no positive overall Pearson correlations)")

    print("\n=== TOP OVERALL (Spearman rho) ===")
    if not top_overall_spearman.empty:
        print(top_overall_spearman[["metric","n","spearman_rho","spearman_p","strong_sig"]].to_string(index=False))
    else:
        print("(no positive overall Spearman correlations)")

    print("\n=== TOP GROUPED (Pearson r) ===")
    if not top_grouped_pearson.empty:
        print(top_grouped_pearson[["metric","group_by","group_value","n","pearson_r","pearson_p","strong_sig"]].to_string(index=False))
    else:
        print("(no positive grouped Pearson correlations)")

    print("\n=== TOP GROUPED (Spearman rho) ===")
    if not top_grouped_spearman.empty:
        print(top_grouped_spearman[["metric","group_by","group_value","n","spearman_rho","spearman_p","strong_sig"]].to_string(index=False))
    else:
        print("(no positive grouped Spearman correlations)")


if __name__ == "__main__":
    main()

In [ ]:
# blend_scan.py
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import pearsonr, spearmanr

# ======================
# CONFIG
# ======================
CSV_PATH = "/content/drive/Shareddrives/Lopez_Morrison_Summer25/code_counts/code_counts.csv"

# Candidate metrics to blend (any missing columns are auto-skipped)
CANDIDATE_METRICS = [
    "total_chars",
    "total_non_ws_chars",
    "token_count",
    "total_python_keywords",
    "max_line_length",
    "cf_keyword_count",
    "avg_line_length",
    "operators_count",
]

TOP_N = 10  # how many top blends to print


# ======================
# Helpers
# ======================
def zscore(series: pd.Series) -> pd.Series:
    std = series.std(ddof=0)
    if std == 0 or np.isnan(std):
        # Return zeros so this metric doesn't blow up the mean
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.mean()) / std


def corr_stats(x: pd.Series, y: pd.Series):
    valid = ~x.isna() & ~y.isna()
    if valid.sum() < 2 or x[valid].nunique() < 2 or y[valid].nunique() < 2:
        return np.nan, np.nan, np.nan, np.nan
    pr, pp = pearsonr(x[valid], y[valid])
    sr, sp = spearmanr(x[valid], y[valid])
    return pr, pp, sr, sp


# ======================
# Main
# ======================
def main():
    df = pd.read_csv(CSV_PATH)

    # Ensure numeric
    if "question_number" not in df.columns:
        raise ValueError("Column 'question_number' not found in CSV.")
    df["question_number"] = pd.to_numeric(df["question_number"], errors="coerce")

    metrics_present = []
    for m in CANDIDATE_METRICS:
        if m in df.columns:
            df[m] = pd.to_numeric(df[m], errors="coerce")
            metrics_present.append(m)

    if not metrics_present:
        raise ValueError("None of the candidate metrics were found in the CSV.")

    results = []

    # Try all non-empty combinations
    for r in range(1, len(metrics_present) + 1):
        for combo in combinations(metrics_present, r):
            combo = list(combo)
            # Z-score normalize each metric (column-wise)
            z_df = df[combo].apply(zscore, axis=0)

            # Equal-weight composite across available (row-wise mean ignoring NaNs)
            composite = z_df.mean(axis=1, skipna=True)

            pr, pp, sr, sp = corr_stats(df["question_number"], composite)

            results.append({
                "metrics": ", ".join(combo),
                "num_metrics": len(combo),
                "pearson_r": pr,
                "pearson_p": pp,
                "spearman_rho": sr,
                "spearman_p": sp,
            })

    res_df = pd.DataFrame(results)

    # Rank by absolute value (fix: use 'by' as a column name and key=abs)
    top_pearson = res_df.sort_values(
        by="pearson_r",
        key=lambda s: s.abs(),
        ascending=False
    ).head(TOP_N)

    top_spearman = res_df.sort_values(
        by="spearman_rho",
        key=lambda s: s.abs(),
        ascending=False
    ).head(TOP_N)

    # Save all blends
    out_path = "metric_blend_correlations.csv"
    res_df.to_csv(out_path, index=False)

    pd.options.display.float_format = "{:,.3f}".format

    print(f"\nSaved all {len(res_df)} blends to {out_path}")

    print(f"\n=== TOP {TOP_N} Blends by abs(Pearson r) ===")
    print(top_pearson[["metrics", "num_metrics", "pearson_r", "pearson_p"]].to_string(index=False))

    print(f"\n=== TOP {TOP_N} Blends by abs(Spearman rho) ===")
    print(top_spearman[["metrics", "num_metrics", "spearman_rho", "spearman_p"]].to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
# blend_scan_grouped.py
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import pearsonr, spearmanr

# ======================
# CONFIG
# ======================
CSV_PATH = "/content/drive/Shareddrives/Lopez_Morrison_Summer25/code_counts/code_counts.csv"

# Candidate metrics to blend (any missing columns are auto-skipped)
CANDIDATE_METRICS = [
    "total_chars",
    "total_non_ws_chars",
    "token_count",
    "total_python_keywords",
    "max_line_length",
    "cf_keyword_count",
    "avg_line_length",
    "operators_count",
]

GROUP_BYS = ["method", "model", "subject"]  # attributes to split by
MIN_GROUP_N = 6   # minimum rows per group to compute group correlations

TOP_N = 10        # how many top blends to print in leaderboards


# ======================
# Helpers
# ======================
def zscore(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    std = s.std(ddof=0)
    if std == 0 or np.isnan(std):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std

def rankify(s: pd.Series) -> pd.Series:
    # average ranking for ties, normalized not required for Spearman
    return s.rank(method="average")

def corr_stats(x: pd.Series, y: pd.Series):
    valid = ~x.isna() & ~y.isna()
    if valid.sum() < 2 or x[valid].nunique() < 2 or y[valid].nunique() < 2:
        return np.nan, np.nan, np.nan, np.nan
    pr, pp = pearsonr(x[valid], y[valid])
    sr, sp = spearmanr(x[valid], y[valid])
    return pr, pp, sr, sp

def equal_weight_blends(df_scope: pd.DataFrame, metrics_present: list[str]):
    """Yield result rows for every non-empty combination using equal weights."""
    results = []
    q = pd.to_numeric(df_scope["question_number"], errors="coerce")
    for r in range(1, len(metrics_present) + 1):
        for combo in combinations(metrics_present, r):
            combo = list(combo)

            # Z-score normalize each metric within this scope
            Z = df_scope[combo].apply(zscore, axis=0)

            # Equal-weight composite (row-wise mean ignoring NaNs)
            composite = Z.mean(axis=1, skipna=True)

            pr, pp, sr, sp = corr_stats(q, composite)
            results.append({
                "weighting": "equal",
                "metrics": ", ".join(combo),
                "num_metrics": len(combo),
                "pearson_r": pr, "pearson_p": pp,
                "spearman_rho": sr, "spearman_p": sp,
            })
    return results

def corr_weighted_blend(df_scope: pd.DataFrame, metrics_present: list[str]):
    """Correlation-weighted blend using |Pearson r| as weights within scope."""
    q = pd.to_numeric(df_scope["question_number"], errors="coerce")
    # compute weights per metric in scope
    weights = {}
    for m in metrics_present:
        pr, _, _, _ = corr_stats(q, pd.to_numeric(df_scope[m], errors="coerce"))
        weights[m] = abs(pr) if not np.isnan(pr) else 0.0

    wsum = sum(weights.values())
    if wsum == 0:
        # fallback to equal if all zero
        w = {m: 1/len(metrics_present) for m in metrics_present}
    else:
        w = {m: weights[m]/wsum for m in metrics_present}

    # z-score each metric and weight
    Z = df_scope[metrics_present].apply(zscore, axis=0)
    composite = Z.mul(pd.Series(w)).sum(axis=1, skipna=True)

    pr, pp, sr, sp = corr_stats(q, composite)
    return [{
        "weighting": "corr-weighted",
        "metrics": ", ".join(metrics_present),
        "num_metrics": len(metrics_present),
        "pearson_r": pr, "pearson_p": pp,
        "spearman_rho": sr, "spearman_p": sp,
    }]

def ols_optimized_blend(df_scope: pd.DataFrame, metrics_present: list[str]):
    """OLS on z-scored metrics to predict standardized question_number."""
    q_raw = pd.to_numeric(df_scope["question_number"], errors="coerce")
    q = zscore(q_raw)
    X = df_scope[metrics_present].apply(zscore, axis=0)

    # Drop rows where target or all features are NaN
    valid_rows = (~q.isna()) & (~X.isna().all(axis=1))
    if valid_rows.sum() < 2 or X.loc[valid_rows].shape[1] == 0:
        return [{
            "weighting": "ols-optimized",
            "metrics": ", ".join(metrics_present),
            "num_metrics": len(metrics_present),
            "pearson_r": np.nan, "pearson_p": np.nan,
            "spearman_rho": np.nan, "spearman_p": np.nan,
        }]

    Xv = X.loc[valid_rows].fillna(0.0).to_numpy()
    yv = q.loc[valid_rows].to_numpy()

    # Solve least squares: minimize ||Xw - y||
    # Add small ridge term if needed for stability (commented by default)
    # lam = 0.0
    # w = np.linalg.solve(Xv.T @ Xv + lam*np.eye(Xv.shape[1]), Xv.T @ yv)
    w, *_ = np.linalg.lstsq(Xv, yv, rcond=None)

    composite = pd.Series(Xv @ w, index=X.index[valid_rows])

    pr, pp, sr, sp = corr_stats(q.loc[valid_rows], composite)
    return [{
        "weighting": "ols-optimized",
        "metrics": ", ".join(metrics_present),
        "num_metrics": len(metrics_present),
        "pearson_r": pr, "pearson_p": pp,
        "spearman_rho": sr, "spearman_p": sp,
    }]

def ols_optimized_blend_spearman(df_scope: pd.DataFrame, metrics_present: list[str]):
    """OLS on rank-transformed (Spearman-style) data."""
    q_raw = pd.to_numeric(df_scope["question_number"], errors="coerce")
    q = rankify(q_raw)
    # z-score ranks so features comparable
    X = df_scope[metrics_present].apply(lambda col: rankify(pd.to_numeric(col, errors="coerce")), axis=0)
    X = X.apply(zscore, axis=0)
    q = zscore(q)

    valid_rows = (~q.isna()) & (~X.isna().all(axis=1))
    if valid_rows.sum() < 2 or X.loc[valid_rows].shape[1] == 0:
        return [{
            "weighting": "ols-optimized-spearman",
            "metrics": ", ".join(metrics_present),
            "num_metrics": len(metrics_present),
            "pearson_r": np.nan, "pearson_p": np.nan,
            "spearman_rho": np.nan, "spearman_p": np.nan,
        }]

    Xv = X.loc[valid_rows].fillna(0.0).to_numpy()
    yv = q.loc[valid_rows].to_numpy()

    w, *_ = np.linalg.lstsq(Xv, yv, rcond=None)
    composite = pd.Series(Xv @ w, index=X.index[valid_rows])

    pr, pp, sr, sp = corr_stats(q.loc[valid_rows], composite)
    return [{
        "weighting": "ols-optimized-spearman",
        "metrics": ", ".join(metrics_present),
        "num_metrics": len(metrics_present),
        "pearson_r": pr, "pearson_p": pp,
        "spearman_rho": sr, "spearman_p": sp,
    }]

def compute_scope_results(df_scope: pd.DataFrame, scope: str, gb_col: str = "", gb_val: str = ""):
    """Compute all blends for a given scope dataframe."""
    # filter metrics present with enough non-NaNs
    m_present = []
    for m in CANDIDATE_METRICS:
        if m in df_scope.columns and df_scope[m].notna().sum() >= 2:
            m_present.append(m)
    results = []
    if not m_present:
        return results

    # 1) equal-weight for all combinations
    results += equal_weight_blends(df_scope, m_present)

    # 2) correlation-weighted (full set only)
    results += corr_weighted_blend(df_scope, m_present)

    # 3) OLS-optimized (full set only)
    results += ols_optimized_blend(df_scope, m_present)

    # 4) OLS-optimized on rank-transformed (approx Spearman-optimized)
    results += ols_optimized_blend_spearman(df_scope, m_present)

    # Attach scope metadata
    for row in results:
        row["scope"] = scope
        row["group_by"] = gb_col
        row["group_value"] = gb_val
        row["n"] = int(df_scope.dropna(subset=["question_number"]).shape[0])

    return results

def print_leaderboards(df_res: pd.DataFrame, title_suffix: str = ""):
    def _top(df_in, score_col, label):
        d = df_in.dropna(subset=[score_col])
        d = d.sort_values(score_col, key=lambda s: s.abs(), ascending=False).head(TOP_N)
        return d

    print(f"\n=== TOP {TOP_N} (Pearson r){title_suffix} ===")
    top_p = _top(df_res, "pearson_r", "pearson")
    if top_p.empty:
        print("(none)")
    else:
        cols = ["scope","group_by","group_value","weighting","metrics","num_metrics","n","pearson_r","pearson_p"]
        print(top_p[cols].to_string(index=False))

    print(f"\n=== TOP {TOP_N} (Spearman rho){title_suffix} ===")
    top_s = _top(df_res, "spearman_rho", "spearman")
    if top_s.empty:
        print("(none)")
    else:
        cols = ["scope","group_by","group_value","weighting","metrics","num_metrics","n","spearman_rho","spearman_p"]
        print(top_s[cols].to_string(index=False))


# ======================
# Main
# ======================
def main():
    df = pd.read_csv(CSV_PATH)

    if "question_number" not in df.columns:
        raise ValueError("Column 'question_number' not found in CSV.")
    df["question_number"] = pd.to_numeric(df["question_number"], errors="coerce")

    # Coerce candidate metrics to numeric (ignore missing)
    for m in CANDIDATE_METRICS:
        if m in df.columns:
            df[m] = pd.to_numeric(df[m], errors="coerce")

    all_rows = []

    # ---------- Overall scope ----------
    overall_rows = compute_scope_results(df, scope="overall", gb_col="", gb_val="")
    all_rows.extend(overall_rows)

    # ---------- Grouped scopes ----------
    for gb in GROUP_BYS:
        if gb not in df.columns:
            continue
        for gv, gdf in df.groupby(gb, dropna=False):
            # coerce group label to string (including NaN)
            gv_str = "" if pd.isna(gv) else str(gv)
            # enforce minimum group size
            if gdf.dropna(subset=["question_number"]).shape[0] < MIN_GROUP_N:
                continue
            rows = compute_scope_results(gdf, scope=f"by_{gb}", gb_col=gb, gb_val=gv_str)
            all_rows.extend(rows)

    out = pd.DataFrame(all_rows, columns=[
        "scope","group_by","group_value","n",
        "weighting","metrics","num_metrics",
        "pearson_r","pearson_p","spearman_rho","spearman_p"
    ])

    out_path = "metric_blend_correlations_grouped.csv"
    out.to_csv(out_path, index=False)
    print(f"\nSaved: {out_path} (rows={len(out)})")

    # Print leaderboards
    print_leaderboards(out, title_suffix=" • ALL SCOPES")
    print_leaderboards(out[out["scope"]=="overall"], title_suffix=" • OVERALL ONLY")

    # Optional: per-group leaderboards
    for gb in GROUP_BYS:
        sub = out[out["group_by"] == gb]
        if not sub.empty:
            print_leaderboards(sub, title_suffix=f" • grouped by {gb}")

if __name__ == "__main__":
    main()


In [ ]:
My professors requested some sort of metric which computes the simularity between the MCQ answer choices. Something like using a transformer for each of the question choices and then doing cos_simularity between all of them.